# City Predictor Machine-learning Project

Below is extensive testing and protoryping to better understand which model architecture is best fit for the task at hand.
That is, we determine which model is best at determining if a given data point describes **New York City** 🇺🇸, **Dubai** 🇦🇪,
**Rio de Janeiro** 🇧🇷, or **Paris 🇫🇷**. We will be exploring some tranformations of the provided dataset, and test out the performance of various model families.

In [1]:
import csv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt # For plotting

Now, we load the data into pandas:

In [7]:
data = pd.read_csv("data/cleaned_dataset.csv")
del data['id'] #remove id column since it is of no use
data

,Popularity,Virality,Architecure,Parties,Company,Relatable,Temperature,Languages,Styles,Quote,Label
0,4,1.0,2.0,1,Co-worker,"Skyscrapers=>6,Sport=>4,Art and Music=>2,Carni...",25,7.0,100,Slavery,Dubai
1,4,3.0,5.0,2,Co-worker,"Skyscrapers=>6,Sport=>1,Art and Music=>2,Carni...",20,3.0,4,"Wherever there is great property, there is gre...",Dubai
2,5,4.0,5.0,1,"Partner,Friends","Skyscrapers=>6,Sport=>2,Art and Music=>3,Carni...",32,5.0,4,Futuristic land,Dubai
3,5,4.0,4.0,1,"Partner,Friends","Skyscrapers=>6,Sport=>5,Art and Music=>1,Carni...",23,10.0,3,The city where anything is possible,Dubai
4,4,3.0,3.0,3,"Partner,Friends,Siblings","Skyscrapers=>6,Sport=>4,Art and Music=>1,Carni...",20,10.0,5,"If you can think of a high building, it probab...",Dubai
...,...,...,...,...,...,...,...,...,...,...,...
1457,4,2.0,5.0,3,"Partner,Friends,Siblings","Skyscrapers=>2,Sport=>1,Art and Music=>4,Carni...",5,2.0,2,"""The average piece of junk is probably more me...",Paris
1458,5,3.0,4.0,2,"Partner,Co-worker","Skyscrapers=>1,Sport=>3,Art and Music=>6,Carni...",7,2.0,4,Oui oui baguette,Paris
1459,2,1.0,3.0,4,Friends,"Skyscrapers=>2,Sport=>3,Art and Music=>6,Carni...",9,87.0,67,E OhÂ,Paris
1460,5,4.0,5.0,3,"Partner,Friends","Skyscrapers=>5,Sport=>6,Art and Music=>2,Carni...",15,1.0,15,croissants and cigarettesÂ,Paris


Now, since the "relatable" column has too many unique values, and won't provide much insight for any standard models, we split the relatability scores
into seperate columns:

In [8]:
upd_data = pd.read_csv("data/cleaned_dataset_split.csv")
del upd_data['id'] #remove id column since it is of no use
upd_data

,Popularity,Virality,Architecure,Parties,Company,rel_Art and Music,rel_Carnival,rel_Cuisine,rel_Economic,rel_Skyscrapers,rel_Sport,Temperature,Languages,Styles,Quote,Label
0,4,1.0,2.0,1,Co-worker,2,1,3.0,5,6,4,25,7.0,100,Slavery,Dubai
1,4,3.0,5.0,2,Co-worker,2,3,4.0,5,6,1,20,3.0,4,"Wherever there is great property, there is gre...",Dubai
2,5,4.0,5.0,1,"Partner,Friends",3,3,4.0,5,6,2,32,5.0,4,Futuristic land,Dubai
3,5,4.0,4.0,1,"Partner,Friends",1,2,4.0,3,6,5,23,10.0,3,The city where anything is possible,Dubai
4,4,3.0,3.0,3,"Partner,Friends,Siblings",1,2,3.0,5,6,4,20,10.0,5,"If you can think of a high building, it probab...",Dubai
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1457,4,2.0,5.0,3,"Partner,Friends,Siblings",4,6,5.0,3,2,1,5,2.0,2,"""The average piece of junk is probably more me...",Paris
1458,5,3.0,4.0,2,"Partner,Co-worker",6,2,5.0,4,1,3,7,2.0,4,Oui oui baguette,Paris
1459,2,1.0,3.0,4,Friends,6,2,2.0,5,2,3,9,87.0,67,E OhÂ,Paris
1460,5,4.0,5.0,3,"Partner,Friends",2,3,1.0,4,5,6,15,1.0,15,croissants and cigarettesÂ,Paris


Much better! On that note, we also wish to split the "company" column into one-hot columns. That is, we will expand the column into
four distinct _binary_ columns: bring_friends, bring_coworker, bring_partner and bring_siblings. Additionally, to simplify the training
process for now, we will exclude the "Quote" column and add it later on.

In [10]:
complete_data = pd.read_csv("data/cleaned_dataset_complete.csv")
del complete_data['id']
del complete_data['Quote']
complete_data

,Popularity,Virality,Architecure,Parties,bring_Co-worker,bring_Friends,bring_Partner,bring_Siblings,rel_Art and Music,rel_Carnival,rel_Cuisine,rel_Economic,rel_Skyscrapers,rel_Sport,Temperature,Languages,Styles,Label
0,4,1.0,2.0,1,1,0,0,0,2,1,3.0,5,6,4,25,7.0,100,Dubai
1,4,3.0,5.0,2,1,0,0,0,2,3,4.0,5,6,1,20,3.0,4,Dubai
2,5,4.0,5.0,1,0,1,1,0,3,3,4.0,5,6,2,32,5.0,4,Dubai
3,5,4.0,4.0,1,0,1,1,0,1,2,4.0,3,6,5,23,10.0,3,Dubai
4,4,3.0,3.0,3,0,1,1,1,1,2,3.0,5,6,4,20,10.0,5,Dubai
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1457,4,2.0,5.0,3,0,1,1,1,4,6,5.0,3,2,1,5,2.0,2,Paris
1458,5,3.0,4.0,2,1,0,1,0,6,2,5.0,4,1,3,7,2.0,4,Paris
1459,2,1.0,3.0,4,0,1,0,0,6,2,2.0,5,2,3,9,87.0,67,Paris
1460,5,4.0,5.0,3,0,1,1,0,2,3,1.0,4,5,6,15,1.0,15,Paris


Now, we are ready to train the model! We will start with a decision tree since the data at hand is fairly linearly seperable.